Scraper pour Indeed.fr - Collecte d'offres d'emploi via curl_cffi.
curl_cffi impersonne le TLS fingerprint d'un vrai navigateur
pour contourner les protections anti-bot basées sur l'analyse TLS.
Sauvegarde en JSON brut dans data/raw/indeed/

In [ ]:
import os
import re
import json
import time
import random
import logging
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional
from urllib.parse import urlencode

from curl_cffi import requests
from bs4 import BeautifulSoup

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

In [ ]:
BASE_URL = "https://fr.indeed.com/jobs"
RESULTS_PER_PAGE = 15

MIN_DELAY = 5.0
MAX_DELAY = 12.0
MIN_DELAY_DESC = 5.0
MAX_DELAY_DESC = 10.0
PAGE_DELAY_MIN = 15.0
PAGE_DELAY_MAX = 25.0

MAX_CONSECUTIVE_EMPTY = 2

IMPERSONATE_PROFILES = [
    "chrome120",
    "chrome124",
    "chrome131",
    "firefox133",
    "safari17_0",
    "safari17_2_ios",
]

In [ ]:
@dataclass
class JobOffer:
    source: str
    title: str
    company: str
    location: str
    salary: Optional[str]
    contract_type: Optional[str]
    description: Optional[str]
    url: str
    query: str
    scrape_date: str

In [ ]:
def build_session(referer="https://fr.indeed.com/"):
    profile = random.choice(IMPERSONATE_PROFILES)
    session = requests.Session(impersonate=profile)
    session.headers.update({
        "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
        "Referer": referer,
        "DNT": "1",
    })
    return session


def random_delay(min_d=MIN_DELAY, max_d=MAX_DELAY):
    time.sleep(random.uniform(min_d, max_d))


def rotate_profile(session):
    return random.choice(IMPERSONATE_PROFILES)


def is_blocked(soup):
    signals = [
        soup.find(id="challenge-stage"),
        soup.find(id="px-captcha"),
        soup.find("div", class_=re.compile(r"captcha|robot|challenge", re.I)),
    ]
    keywords = [
        "vérifiez que vous n'êtes pas un robot", "verify you are human",
        "access denied", "unusual traffic", "please enable cookies",
        "security check", "checking your browser",
    ]
    text_lower = soup.get_text().lower()
    return any(signals) or any(kw in text_lower for kw in keywords)


def fetch_page(session, url, params, retries=3, referer="https://fr.indeed.com/"):
    session.headers["Referer"] = referer
    for attempt in range(1, retries + 1):
        try:
            profile = rotate_profile(session)
            response = session.get(url, params=params, timeout=15, impersonate=profile)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, "html.parser")
                if is_blocked(soup):
                    logger.warning(f"  Page anti-bot détectée (tentative {attempt}/{retries}). Pause prolongée.")
                    time.sleep(60 * attempt)
                    return None
                return soup
            elif response.status_code == 429:
                logger.warning(f"  Rate limited (429). Attente {60 * attempt}s (tentative {attempt}/{retries}).")
                time.sleep(60 * attempt)
            elif response.status_code in (403, 404):
                logger.warning(f"  Statut HTTP inattendu : {response.status_code} (tentative {attempt}/{retries})")
                time.sleep(10 * attempt)
            else:
                logger.warning(f"  Statut HTTP inattendu : {response.status_code} (tentative {attempt}/{retries})")
                time.sleep(5)
        except Exception as e:
            logger.error(f"  Erreur réseau (tentative {attempt}/{retries}) : {e}")
            time.sleep(5 * attempt)
    return None

In [ ]:
def parse_salary(card):
    selectors = [{"data-testid": "attribute_snippet_testid"},
                 "div.salary-snippet-container", "div.metadata.salary-snippet-container"]
    for sel in selectors:
        el = card.find("div", attrs=sel) if isinstance(sel, dict) else card.select_one(sel)
        if el:
            text = el.get_text(strip=True)
            if any(c in text for c in ["\u20ac", "$", "\u00a3", "par an", "par mois"]):
                return text
    return None


def parse_contract_type(card):
    el = card.find("div", attrs={"data-testid": "attribute_snippet_testid"})
    if not el:
        for meta in card.find_all("div", class_=re.compile(r"metadata")):
            text = meta.get_text(strip=True)
            for kw in ["CDI", "CDD", "Stage", "Alternance", "Freelance",
                       "Intérim", "Temps plein", "Temps partiel"]:
                if kw.lower() in text.lower():
                    return text
    elif el:
        return el.get_text(strip=True)
    return None


def parse_job_cards(soup, query, base_url_prefix="https://fr.indeed.com"):
    offers = []
    today = datetime.now().strftime("%Y-%m-%d")
    cards = soup.find_all("div", class_="job_seen_beacon")
    if not cards:
        cards = soup.find_all("div", attrs={"data-testid": "slider_item"})
    logger.info(f"  {len(cards)} cartes trouvées sur cette page")
    for card in cards:
        try:
            title_el = card.find("h2", class_=re.compile(r"jobTitle")) or card.find("a", attrs={"data-jk": True})
            title = title_el.get_text(strip=True) if title_el else "N/A"
            company_el = card.find("span", attrs={"data-testid": "company-name"}) or \
                         card.find("a", attrs={"data-testid": "company-name"})
            company = company_el.get_text(strip=True) if company_el else "N/A"
            location_el = card.find("div", attrs={"data-testid": "text-location"})
            location = location_el.get_text(strip=True) if location_el else "N/A"
            link_el = card.find("a", attrs={"data-jk": True})
            job_url = (base_url_prefix + link_el["href"]) if link_el else "N/A"
            desc_el = card.find("div", attrs={"data-testid": "jobsnippet_footer"}) or \
                      card.find("div", class_=re.compile(r"job-snippet|underShelfFooter"))
            offers.append(JobOffer(
                source="indeed", title=title, company=company, location=location,
                salary=parse_salary(card), contract_type=parse_contract_type(card),
                description=desc_el.get_text(separator=" ", strip=True) if desc_el else None,
                url=job_url, query=query, scrape_date=today,
            ))
        except Exception as e:
            logger.debug(f"  Erreur parsing carte : {e}")
    return offers


def fetch_job_description(session, job_url, search_page_url=BASE_URL):
    if job_url == "N/A":
        return None
    soup = fetch_page(session, job_url, params={}, referer=search_page_url)
    if not soup:
        return None
    desc_el = soup.find("div", attrs={"id": "jobDescriptionText"}) or \
               soup.find("div", class_=re.compile(r"jobsearch-jobDescriptionText"))
    return desc_el.get_text(separator="\n", strip=True) if desc_el else None


def scrape_indeed(query, location, max_pages=5, fetch_descriptions=True, output_dir=None):
    if output_dir is None:
        output_dir = os.path.join("..", "data", "raw", "indeed")
    os.makedirs(output_dir, exist_ok=True)

    session = build_session(referer="https://fr.indeed.com/")
    all_offers, consecutive_empty = [], 0
    logger.info(f"Recherche Indeed : '{query}' à '{location}' ({max_pages} pages)")

    for page in range(max_pages):
        start = page * RESULTS_PER_PAGE
        params = {"q": query, "l": location, "start": start, "lang": "fr"}
        logger.info(f"  Page {page + 1}/{max_pages} (start={start})")

        if page > 0 and page % 2 == 0:
            logger.info("  Renouvellement de session (cookies + profil d'impersonation).")
            session = build_session(referer=BASE_URL)
            random_delay(PAGE_DELAY_MIN, PAGE_DELAY_MAX)

        current_search_url = f"{BASE_URL}?{urlencode(params)}"
        soup = fetch_page(session, BASE_URL, params, referer="https://fr.indeed.com/")

        if soup is None:
            logger.warning(f"  Impossible de charger la page {page + 1}, page ignorée.")
            consecutive_empty += 1
            if consecutive_empty >= MAX_CONSECUTIVE_EMPTY:
                logger.warning(f"  {MAX_CONSECUTIVE_EMPTY} pages non exploitables : arrêt du scraping.")
                break
            continue

        if soup.find("div", class_=re.compile(r"no_results|noResultsSection")):
            logger.info("  Fin des résultats atteinte.")
            break

        page_offers = parse_job_cards(soup, query)
        if not page_offers:
            consecutive_empty += 1
            logger.warning(f"  Aucune offre parsée [{consecutive_empty}/{MAX_CONSECUTIVE_EMPTY}]")
            if consecutive_empty >= MAX_CONSECUTIVE_EMPTY:
                logger.warning(f"  {MAX_CONSECUTIVE_EMPTY} pages vides consécutives : arrêt du scraping.")
                break
        else:
            consecutive_empty = 0

        if fetch_descriptions and page_offers:
            for i, offer in enumerate(page_offers):
                logger.info(f"    Description {i + 1}/{len(page_offers)} : {offer.title[:50]}")
                full_desc = fetch_job_description(session, offer.url, current_search_url)
                if full_desc:
                    offer.description = full_desc
                random_delay(MIN_DELAY_DESC, MAX_DELAY_DESC)

        all_offers.extend(page_offers)
        logger.info(f"  Total cumulé : {len(all_offers)} offres")

        if page < max_pages - 1:
            delay = random.uniform(PAGE_DELAY_MIN, PAGE_DELAY_MAX)
            logger.info(f"  Pause inter-page : {delay:.1f}s")
            time.sleep(delay)

    if all_offers:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_query = re.sub(r"[^\w]", "_", query)
        filepath = os.path.join(output_dir, f"indeed_{safe_query}_{timestamp}.json")
        data = [asdict(o) for o in all_offers]
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        logger.info(f"{len(all_offers)} offres sauvegardées : {filepath}")
    else:
        logger.warning("Aucune offre collectée.")
        data = []
    return data

## Lancement du scraper

In [ ]:
results = scrape_indeed(
    query="data engineer",
    location="Paris",
    max_pages=5,
    fetch_descriptions=True,
)
print(f"Scraping terminé : {len(results)} offres collectées.")